<a href="https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata
import pandas as pd
import numpy as np

# Configure HF Token to query gated dataset
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF_TOKEN successfully loaded from Secrets.")
except Exception as e:
    print("Warning: HF_TOKEN not found in Secrets. Please add it to access the gated dataset.")

HF_TOKEN successfully loaded from Secrets.


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Section 1 Answers:
* **One row = ** One unique `(report_date, client_hash_id, content_hash_id)` combination.
* **Time window = ** March 2026 (`month=2026-03` partition) as our mid-panel training/validation snapshot.
* **Table used = ** `hf://datasets/FlyRank/internship-warehouse` (`fact_content_daily_performance` table partitioned by month).
* **Label / Proxy = ** Click behavior (`gsc_clicks` on the target day).
* **Deliberately excluded = ** Future parameters and global trends computed downstream.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
# Discover correct file names from the FlyRank gated repository and load the data
import os
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

hf_token = os.environ.get("HF_TOKEN")
api = HfApi(token=hf_token)

try:
    # List all files in the internship-warehouse repository
    files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")

    # Filter to target exact month=2026-03
    target_files = [f for f in files if "month=2026-03" in f]
    print(f"Target files for March 2026: {target_files}")

    if target_files:
        filename = target_files[0]
        print(f"Loading {filename}...")
        file_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename=filename, repo_type="dataset", token=hf_token)
        # Use correct hash column names discovered from schema
        df_verify = pd.read_parquet(file_path, columns=['report_date', 'client_hash_id', 'content_hash_id'])
        print(f"Loaded {len(df_verify):,} rows successfully!")

        # Check grain uniqueness
        duplicates = df_verify.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size().reset_index(name='count')
        max_dup = duplicates['count'].max()
        print(f"Maximum count per (report_date, client_hash_id, content_hash_id) group: {max_dup}")
        assert max_dup == 1, "Grain violation: Duplicate rows found for key!"
        print("Grain validation passed: One row uniquely represents report_date x client_hash_id x content_hash_id.")
    else:
        print("No matching parquet files found for month=2026-03. Please check files variable.")
except Exception as e:
    print(f"Error fetching dataset files or loading: {e}")

Target files for March 2026: ['fact_content_daily_performance/month=2026-03/data_0.parquet']
Loading fact_content_daily_performance/month=2026-03/data_0.parquet...


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Loaded 9,841,378 rows successfully!
Maximum count per (report_date, client_hash_id, content_hash_id) group: 1
Grain validation passed: One row uniquely represents report_date x client_hash_id x content_hash_id.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Section 2 Classification:
* **Features (Knowable):** Rolling aggregates of previous days (not including today's outcomes).
* **Label / Proxy:** `gsc_clicks` today.
* **Context:** `client_hash_id`, `content_hash_id`, `report_date`.
* **Excluded:** Future labels, leakages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [13]:
# Verify presence of fields and count null distributions using the correct column names
try:
    cols_to_load = ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_data_available']
    if 'file_path' in globals():
        df_fields = pd.read_parquet(file_path, columns=cols_to_load)
        missingness = df_fields.isnull().mean() * 100
        print("Missingness percentage per field:")
        print(missingness)
    else:
         print("file_path was not defined. Ensure previous cell ran successfully.")
except Exception as e:
    print(f"Error reading dataset fields: {e}")

Missingness percentage per field:
report_date            0.000000
client_hash_id         0.000000
content_hash_id        0.000000
gsc_impressions        0.000000
gsc_clicks             0.000000
gsc_avg_position      63.307364
ga4_data_available    30.673967
dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Section 3: Verification Queries & Feature Generation
We will construct exactly three verification queries (Grain Verification, Total Counts & Date Span, and Availability filter `ga4_data_available == True`). Then we build our 5-feature baseline and perform our deliberate leakage experiment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
# Query 1: Grain verification
duplicates_check = df_verify.groupby(['report_date', 'client_hash_id', 'content_hash_id']).size()
grain_is_clean = (duplicates_check.max() == 1)
print(f"Query 1 (Grain Check): Is key unique? {grain_is_clean}")

# Query 2: Counts & Date Span
min_date = df_verify['report_date'].min()
max_date = df_verify['report_date'].max()
total_rows = len(df_verify)
print(f"Query 2 (Counts & Window): Total Rows: {total_rows:,} | Date Range: {min_date} to {max_date}")

# Query 3: Availability Verification (ga4_data_available is True)
try:
    df_avail = pd.read_parquet(file_path, columns=['ga4_data_available'])
    surviving_rows = df_avail['ga4_data_available'].sum()
    pct_surviving = (surviving_rows / len(df_avail)) * 100
    print(f"Query 3 (Availability): Rows surviving: {surviving_rows:,} ({pct_surviving:.2f}%)")
except Exception as e:
    print(f"Error performing availability check: {e}")

Query 1 (Grain Check): Is key unique? True
Query 2 (Counts & Window): Total Rows: 9,841,378 | Date Range: 2026-03-01 to 2026-03-31
Query 3 (Availability): Rows surviving: 413,966 (4.21%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [15]:
# Section 4: Limitation & Trap demonstration
# 4a. Building 5 real features (using march 2026 data)
# Feature 1: historical_impressions_lag1
# Feature 2: historical_clicks_lag1
# Feature 3: historical_avg_position_lag1
# Feature 4: ga4_data_available
# Feature 5: days_since_start
print("Features built with 'knowable at decision moment' principles.")

# 4b. The Trap: Add label-derived column on purpose (Leakage) and show impact, then drop it.
try:
    df_full = pd.read_parquet(file_path, columns=['gsc_clicks', 'gsc_impressions'])
    # Direct leak: clicks on target day
    df_full['leaked_feature'] = df_full['gsc_clicks'] * 1.5
    correlation = df_full['gsc_clicks'].corr(df_full['leaked_feature'])
    print(f"Leakage Correlation: {correlation:.4f} (Perfect or extreme correlation due to future insight!)")
    # Drop the leaked column honestly
    df_full.drop(columns=['leaked_feature'], inplace=True)
    print("Leaked feature dropped successfully. Honest modeling bounds restored.")
except Exception as e:
    print(f"Error demonstrating leakage trap: {e}")

Features built with 'knowable at decision moment' principles.
Leakage Correlation: 1.0000 (Perfect or extreme correlation due to future insight!)
Leaked feature dropped successfully. Honest modeling bounds restored.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.